# Train `hey_nova` wake word

1. **Runtime > Change runtime type > T4 GPU**
2. **Runtime > Run all**
3. The last cell downloads `hey_nova.onnx` -> put it in `wakeword/models/`.

# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup (Colab GPU runtime) - safe to re-run
import os, sys
!rm -rf piper-sample-generator openwakeword

# piper-sample-generator: must be the v2.0.0 tag (master was restructured and has no generate_samples.py)
!git clone -q --branch v2.0.0 https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# piper-phonemize has no wheel for Colab's Python 3.13, so shim the one function used with espeak-ng
!apt-get -qq install -y espeak-ng > /dev/null
with open("piper-sample-generator/piper_phonemize.py", "w") as f:
    f.write("""import subprocess, unicodedata
def phonemize_espeak(text, voice):
    out = subprocess.run(["espeak-ng", "-v", voice, "-q", "--ipa"], input=text,
                         capture_output=True, text=True, check=True).stdout
    return [list(unicodedata.normalize("NFD", " ".join(out.split())))]
""")

!pip install -q webrtcvad-wheels librosa pytorch-lightning

!git clone -q https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword --no-deps
!pip install -q onnxruntime pyyaml scikit-learn requests tqdm scipy

# Training deps. Unpinned (old pins don't build on current Colab). TensorFlow / onnx_tf are
# intentionally skipped: we only need the .onnx, which train.py saves before the tflite step.
!pip install -q mutagen torchinfo torchmetrics speechbrain audiomentations torch-audiomentations acoustics pronouncing 'datasets<4' deep-phonemizer onnx

# Pretrained feature models openWakeWord needs
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
for f in ["embedding_model.onnx","embedding_model.tflite","melspectrogram.onnx","melspectrogram.tflite"]:
    !wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{f} -O ./openwakeword/openwakeword/resources/models/{f}

import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"

# ---- Compatibility patches for current Colab (torch >= 2.6, torchaudio >= 2.9) ----
!pip install -q onnxscript soundfile
# 1. piper checkpoint is a full pickled model; torch>=2.6 defaults to weights_only=True
!sed -i 's/torch.load(model_path)/torch.load(model_path, weights_only=False)/' piper-sample-generator/generate_samples.py
# 2. torchaudio >= 2.9 removed info() and load() needs torchcodec: shim both with soundfile
with open("openwakeword/openwakeword/oww_shim.py", "w") as f:
    f.write("""import types, torch, torchaudio, soundfile as sf
def _load(uri, frame_offset=0, num_frames=-1, normalize=True, channels_first=True, **k):
    d, sr = sf.read(str(uri), start=frame_offset, frames=num_frames, dtype="float32", always_2d=True)
    t = torch.from_numpy(d.T.copy()); return (t if channels_first else t.T), sr
def _info(uri, **k):
    i = sf.info(str(uri)); return types.SimpleNamespace(num_frames=i.frames, sample_rate=i.samplerate, num_channels=i.channels)
torchaudio.load = _load; torchaudio.info = _info
""")
!sed -i '1i import oww_shim' openwakeword/openwakeword/train.py
# 3. use the legacy ONNX exporter (dynamo exporter is the new torch default)
!sed -i 's/opset_version=13)/opset_version=13, dynamo=False)/' openwakeword/openwakeword/train.py
# 4. feature extraction: CPU onnxruntime with all cores
!sed -i 's/device="gpu" if torch.cuda.is_available() else "cpu"/device="cpu"/; s/ncpu=n_cpus if not torch.cuda.is_available() else 1/ncpu=os.cpu_count()/' openwakeword/openwakeword/train.py
# 5. audiomentations wants float32
!sed -i 's/dat = np.zeros(n_samples)$/dat = np.zeros(n_samples, dtype=np.float32)/' openwakeword/openwakeword/data.py
!grep -c "weights_only=False" piper-sample-generator/generate_samples.py; grep -c "dynamo=False" openwakeword/openwakeword/train.py; grep -c "dtype=np.float32" openwakeword/openwakeword/data.py


In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm

import scipy.io.wavfile


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
## Download noise and background audio
# The original HF tar / script-based FMA loaders are gone, so read one parquet shard of each
# dataset directly and convert clips to 16 kHz wav.
import io, librosa, pyarrow.parquet as pq

def parquet_to_wavs(url, parquet_path, output_dir, max_clips, max_seconds=None):
    os.makedirs(output_dir, exist_ok=True)
    if not os.path.exists(parquet_path):
        !wget -q -O {parquet_path} {url}
    n = 0
    for batch in pq.ParquetFile(parquet_path).iter_batches(batch_size=16, columns=["audio"]):
        for a in batch.column("audio").to_pylist():
            try:
                y, _ = librosa.load(io.BytesIO(a["bytes"]), sr=16000, mono=True, duration=max_seconds)
            except Exception as e:
                print("skip clip:", e); continue
            scipy.io.wavfile.write(os.path.join(output_dir, f"{n:05d}.wav"), 16000, (y*32767).astype(np.int16))
            n += 1
            if n >= max_clips: return n
    return n

# AudioSet (balanced train, shard 09): ~10 s clips. 2000 clips ~ 5.5 h. Raise for better results.
print(parquet_to_wavs("https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train/09.parquet",
                      "audioset_09.parquet", "./audioset_16k", max_clips=2000), "audioset clips")

# Free Music Archive (small), shard 0: 240 x 30 s = 2 hours of music
print(parquet_to_wavs("https://huggingface.co/datasets/benjamin-paine/free-music-archive-small/resolve/main/data/train-00000-of-00015.parquet",
                      "fma_00.parquet", "./fma", max_clips=240, max_seconds=30), "fma clips")

In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

In [ ]:
# Delete clips/features from any earlier (smaller) run, otherwise generation is skipped as "already enough"
!rm -rf my_custom_model
# Modify values in the config and save a new version
config["target_phrase"] = ["hey nova"]
config["model_name"] = "hey_nova"
config["n_samples"] = 40000
config["n_samples_val"] = 4000
config["tts_batch_size"] = 100
config["steps"] = 60000
config["target_accuracy"] = 0.7
config["target_recall"] = 0.5
config["target_false_positives_per_hour"] = 0.3
config["max_negative_weight"] = 2000
config["custom_negative_phrases"] = ["hey noah", "hey novo", "hey nora", "hey nobody", "hey nolan",
                                     "hey over", "hey mama", "no va", "a nova", "hey now", "hey novak"]

config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: Train model (GPU). The tflite conversion at the very end may error because we skipped
# TensorFlow; that's fine as long as the .onnx below exists.
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model
assert os.path.exists("my_custom_model/hey_nova.onnx"), "hey_nova.onnx missing - check the log above"

In [ ]:
# Step 4: Download the model (save it as wakeword/models/hey_nova.onnx)
from google.colab import files
files.download("my_custom_model/hey_nova.onnx")

After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!